# LAB | Intro to Machine Learning

**Load the data**

In this challenge, we will be working with Spaceship Titanic data. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [12]:
# Step 0 - Import libraries and dataset
# -------------------------------------

# Import the libraries needed for this section
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Standard seed used throughout the notebook, for reproducibility
SEED = 32


In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [4]:
spaceship.shape

(8693, 14)

**Check for data types**

In [5]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [6]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [7]:
spaceship = spaceship.dropna()

**KNN**

K Nearest Neighbors is a distance based algorithm, and requeries all **input data to be numerical.**

Let's only select numerical columns as our features.

And also lets define our target.

In [9]:
X = spaceship[["Age", "FoodCourt", "ShoppingMall", "RoomService", "VRDeck","Spa"]]
y = spaceship["Transported"]

**Train Test Split**

Now that we have split the data into **features** and **target** variables and imported the **train_test_split** function, split X and y into X_train, X_test, y_train, and y_test. 80% of the data should be in the training set and 20% in the test set.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (5284, 6)
X_test shape:  (1322, 6)


In [11]:
# Let's look at the survival (1) rate:
print('Transported y_train:')
print(y_train.value_counts(normalize=True))
print()
print('Transported  in y_test:')
print(y_test.value_counts(normalize=True))

Transported y_train:
Transported
True     0.501893
False    0.498107
Name: proportion, dtype: float64

Transported  in y_test:
Transported
True     0.51059
False    0.48941
Name: proportion, dtype: float64


In [13]:
# Step 2: Train / Test Split
# ---------------------------
# Train / Test Split (without stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (5284, 6)
X_test shape:  (1322, 6)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

You need to choose between **Classificator** or **Regressor**. Take into consideration target variable to decide.

Initialize a KNN instance without setting any hyperparameter.

Fit the model to your data.

In [14]:
# Step 3 - Feature Engineering: Feature Scaling
# ---------------------------------------------

# KNN is distance-based --> standardize with StandardScaler
scaler = StandardScaler()
scaler.set_output(transform="pandas")
# set_output(transform="pandas") makes the scaler return a DataFrame (keeps the real column names and index)

# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
          Age  FoodCourt  ShoppingMall  RoomService     VRDeck        Spa
mean  29.0369   476.2631      172.4769     223.3579   309.9805   312.8229
std   14.6145  1626.3909      545.5146     645.8986  1098.2837  1141.6156

AFTER scaling (X_train):
         Age  FoodCourt  ShoppingMall  RoomService  VRDeck     Spa
mean -0.0000    -0.0000       -0.0000       0.0000 -0.0000  0.0000
std   1.0001     1.0001        1.0001       1.0001  1.0001  1.0001


In [15]:
# Step 4 - Model Training: KNeighborsClassifier
# ---------------------------------------------

# n_neighbors=5 + weights='distance' --> avoids ties + closer neighbors count more (more robust)
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_model.fit(X_train_scaled_df, y_train)  # Fit on train

print('Model trained!')
# KNN does NOT learn coefficients (no coef_) --> it just memorizes the training points and their distances!!!

Model trained!


Evaluate your model.

In [19]:
# Step 5 - Results
# ----------------

# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = knn_model.predict(X_train_scaled_df)  # train
y_pred_test = knn_model.predict(X_test_scaled_df)    # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 89.91%

              precision    recall  f1-score   support

       False       0.79      0.71      0.75       647
        True       0.75      0.82      0.78       675

    accuracy                           0.77      1322
   macro avg       0.77      0.76      0.76      1322
weighted avg       0.77      0.77      0.76      1322



**Congratulations, you have just developed your first Machine Learning model!**